# Multimodal Models 2: Text-Conditioned Diffusion and Image Generation

## DALL·E 2, Imagen, Stable Diffusion—and a compact PyTorch case study

Diffusion models learn to reverse a gradual noising process. Given clean image $x_0$, the forward process forms $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$. A denoiser predicts noise from the noisy image, timestep, and conditioning signal. Classifier-free guidance combines conditional and unconditional predictions to strengthen prompt alignment.

Three influential systems place the text and image representations differently. **DALL·E 2** uses a prior to map text to a CLIP image embedding, followed by a decoder that generates an image from that embedding. **Imagen** uses a frozen T5 text encoder and cascaded pixel-space diffusion/super-resolution models. **Stable Diffusion** performs denoising in a compressed VAE latent space and injects text conditioning through cross-attention. These are distinct design choices, not interchangeable names for one architecture.

This notebook trains a tiny class-conditioned DDPM on CIFAR-10 images (or a procedural fallback). Class IDs act as a deliberately restricted text condition; this demonstrates conditional denoising but is not natural-language generation and does not reproduce those large systems. Training from scratch is kept short for CPU/GPU feasibility.

## 1. Load small image-condition pairs

CIFAR-10 labels provide compact prompt concepts such as “cat” or “ship”. We use a small subset and resize nothing because the source is already 32×32. If torchvision or data download fails, colored geometric icons provide a fallback with the same image/class interface.

**What this cell does:** loads the dataset, creates a training subset, normalizes images to `[-1,1]`, and defines a batch loader.

In [ ]:
import random,numpy as np,torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,Subset
import matplotlib.pyplot as plt
SEED=9;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); labels=['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
try:
 from torchvision.datasets import CIFAR10
 from torchvision import transforms
 base=CIFAR10('./data',train=True,download=True,transform=transforms.ToTensor())
except Exception as e:
 print('Using procedural image fallback:',type(e).__name__)
 class Icons(Dataset):
  def __len__(self):return 1200
  def __getitem__(self,i):
   y=i%10;x=torch.rand(3,32,32)*.08;x[y%3,8:24,8:24]+=.8;return x.clamp(0,1),y
 base=Icons()
ids=np.random.permutation(len(base))[:min(1600,len(base))]; subset=Subset(base,ids)
class Normalized(Dataset):
 def __len__(self):return len(subset)
 def __getitem__(self,i):x,y=subset[i];return x*2-1,int(y)
loader=DataLoader(Normalized(),batch_size=64,shuffle=True,drop_last=True)
print('examples',len(subset),'device',device)


## 2. Noise schedule and conditional denoiser

We use the standard variance-preserving schedule and train with mean squared error between true and predicted noise. The compact convolutional denoiser receives a timestep embedding and a class embedding; their sum conditions each feature map. A null class ID is reserved for classifier-free guidance.

**What this cell does:** defines a time embedding, small U-Net-like down/up path, noise schedule, and forward noising operation.

In [ ]:
T=100;NCLASS=10;NULL=NCLASS;BETAS=torch.linspace(1e-4,.02,T,device=device);ALPHA=1-BETAS;ABAR=ALPHA.cumprod(0)
def embed_t(t,dim=64):
 half=dim//2;f=torch.exp(torch.arange(half,device=t.device)*(-np.log(10000)/(half-1)));a=t.float()[:,None]*f[None,:];return torch.cat([a.sin(),a.cos()],-1)
class TinyDenoiser(nn.Module):
 def __init__(self):
  super().__init__();self.cls=nn.Embedding(NCLASS+1,64);self.temb=nn.Sequential(nn.Linear(64,128),nn.SiLU(),nn.Linear(128,64));self.cond=nn.Linear(64,128)
  self.c1=nn.Conv2d(3,64,3,1,1);self.d=nn.Conv2d(64,128,4,2,1);self.mid=nn.Conv2d(128,128,3,1,1);self.u=nn.ConvTranspose2d(128,64,4,2,1);self.out=nn.Conv2d(64,3,3,1,1)
 def forward(self,x,t,y):
  cond=self.cond(self.cls(y)+self.temb(embed_t(t)));h=F.silu(self.c1(x));d=F.silu(self.d(h)+cond[:,:,None,None]);m=F.silu(self.mid(d));u=F.silu(self.u(m)+h);return self.out(u)
model=TinyDenoiser().to(device);print('parameters',sum(p.numel() for p in model.parameters()))
def q_sample(x0,t,noise):
 a=ABAR[t][:,None,None,None];return a.sqrt()*x0+(1-a).sqrt()*noise


## 3. Train the conditional DDPM

Classifier-free training occasionally replaces the class condition with a null token. At sampling time, the unconditional and conditional noise predictions are combined as $\epsilon_u+w(\epsilon_c-\epsilon_u)$. The guidance scale trades diversity for stronger class adherence.

**What this cell does:** trains for a small number of epochs, sampling random diffusion timesteps and optimizing the noise-prediction objective.

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=2e-4);EPOCHS=5;hist=[]
for ep in range(EPOCHS):
 model.train();ls=[]
 for x,y in loader:
  x=x.to(device);y=y.to(device);t=torch.randint(T,(len(x),),device=device);eps=torch.randn_like(x);xt=q_sample(x,t,eps)
  cond=y.clone();cond[torch.rand(len(y),device=device)<.1]=NULL;pred=model(xt,t,cond);loss=F.mse_loss(pred,eps)
  opt.zero_grad(set_to_none=True);loss.backward();opt.step();ls.append(loss.item())
 hist.append(float(np.mean(ls)));print(f'epoch {ep+1}/{EPOCHS} noise MSE={hist[-1]:.4f}')
plt.plot(hist,marker='o');plt.xlabel('epoch');plt.ylabel('noise MSE');plt.grid();plt.show()


## 4. Reverse diffusion and conditional samples

At each reverse step, the model estimates noise and the DDPM update moves toward a cleaner image. The sampler below uses classifier-free guidance and a shortened 50-step schedule for speed. The model has only a few epochs of training, so images may remain noisy or class-ambiguous.

**What this cell does:** samples one image for each selected CIFAR-10 concept and displays the outputs.

In [ ]:
@torch.no_grad()
def sample(class_id,guidance=2.0,steps=50):
 model.eval();x=torch.randn(1,3,32,32,device=device);schedule=torch.linspace(T-1,0,steps).long()
 for ti in schedule:
  t=torch.full((1,),int(ti),device=device,dtype=torch.long);yc=torch.tensor([class_id],device=device);yu=torch.tensor([NULL],device=device)
  eps=model(x,t,yu)+guidance*(model(x,t,yc)-model(x,t,yu));ab=ABAR[t][:,None,None,None];x0=(x-torch.sqrt(1-ab)*eps)/torch.sqrt(ab)
  prev=int(max(0,int(ti)-T//steps));ab_prev=ABAR[prev] if ti>0 else torch.tensor(1.,device=device)
  x=torch.sqrt(ab_prev)*x0+torch.sqrt(1-ab_prev)*eps
 return x.clamp(-1,1).add(1).div(2)
fig,ax=plt.subplots(1,6,figsize=(12,2.5))
for i,a in enumerate(ax):a.imshow(sample(i).squeeze().cpu().permute(1,2,0));a.set_title(labels[i],fontsize=8);a.axis('off')
plt.tight_layout();plt.show()


## Key takeaways

- Diffusion learns to predict noise at many corruption levels and generates by iterative denoising.
- DALL·E 2 uses a text-to-CLIP-image-embedding prior plus an image decoder; Imagen uses T5 conditioning and cascaded diffusion; Stable Diffusion denoises in VAE latent space with cross-attention.
- This notebook's class-conditioned DDPM is a small teaching model, not a reproduction of any of those systems.
- Classifier-free guidance strengthens conditioning but can reduce diversity and does not fix weak training data.
- Large text-to-image systems require pretrained components, extensive data, substantial compute, and careful safety evaluation.

**Primary references:** [DALL·E 2](https://arxiv.org/abs/2204.06125), [Imagen](https://arxiv.org/abs/2205.11487), [Latent Diffusion / Stable Diffusion foundation](https://arxiv.org/abs/2112.10752).